# Preparación de los datos EAL 2022-2024

El resultado usa formato largo: cada fila representa un año, tema, corte, competencia e indicador. Este formato facilita filtros, agrupaciones, tablas dinámicas y gráficos sin mantener columnas distintas para cada año o categoría.

In [7]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

## Carga Datos

In [8]:
directorio_actual = Path.cwd().resolve()
raiz_proyecto = directorio_actual.parent

carpeta_entrada = raiz_proyecto / "Data" / "Encuesta_EAL" / "Raw_2022-2024"
carpeta_salida = raiz_proyecto / "Data" / "Encuesta_EAL" / "processed"
fichero_salida = carpeta_salida / "Tablas_EAL_Competencies_2022-2024.csv"

nombres_entrada = [f"Tablas_EAL_{anio}.xlsx" for anio in (2022, 2023, 2024)]
archivos_entrada = [carpeta_entrada / nombre for nombre in nombres_entrada]
archivos_faltantes = [archivo.name for archivo in archivos_entrada if not archivo.exists()]

print(f"Archivos encontrados: {len(archivos_entrada)}")
for archivo in archivos_entrada:
    print(" -", archivo.name)

Archivos encontrados: 3
 - Tablas_EAL_2022.xlsx
 - Tablas_EAL_2023.xlsx
 - Tablas_EAL_2024.xlsx


## Reglas de normalización

Las etiquetas originales contienen espacios finales y dos descripciones diferentes para la competencia técnica. Las limpiamos y asignamos un identificador estable para poder comparar años y tablas.

Se mantienen también el archivo y la hoja de origen: así cualquier cifra puede rastrearse hasta la tabla raw.

In [9]:
COMPETENCIAS = {
    "De dirección": ("direccion", "Dirección"),
    "De trabajo en equipo": ("trabajo_equipo", "Trabajo en equipo"),
    "De atención al público/ trato a clientes": ("atencion_clientes", "Atención al público / trato a clientes"),
    "Administrativas de oficina": ("administrativas_oficina", "Administrativas de oficina"),
    "De resolución de problemas (localización de problemas o fallos, análisis de sus causas y búsqueda de soluciones)": ("resolucion_problemas", "Resolución de problemas"),
    "En lenguas extranjeras": ("lenguas_extranjeras", "Lenguas extranjeras"),
    "Básicas de cálculo y/o comunicación oral o escrita": ("basicas_calculo_comunicacion", "Básicas de cálculo y/o comunicación oral o escrita"),
    "Generales de tecnologías de la información": ("tic_generales", "Generales de tecnologías de la información"),
    "Profesionales de tecnologías de la información": ("tic_profesionales", "Profesionales de tecnologías de la información"),
    "Competencias técnicas, prácticas y otras específicas del puesto de trabajo": ("tecnicas_puesto", "Competencias técnicas del puesto de trabajo"),
    "Técnicas específicas del puesto de trabajo": ("tecnicas_puesto", "Competencias técnicas del puesto de trabajo"),
    "Otras": ("otras", "Otras"),
}

CORTES_EAL16 = {
    "EAL-16": ("total", "Total"),
    "EAL-16a": ("tamano_empresa", "5 a 49 trabajadores"),
    "EAL-16b": ("tamano_empresa", "50 a 499 trabajadores"),
    "EAL-16c": ("tamano_empresa", "Más de 499 trabajadores"),
    "EAL-16d": ("sector", "Industria"),
    "EAL-16e": ("sector", "Construcción"),
    "EAL-16f": ("sector", "Servicios"),
}

CORTES_EAL21 = {
    "TOTAL": ("total", "Total"),
    "INDUSTRIA": ("sector", "Industria"),
    "CONSTRUCCIÓN": ("sector", "Construcción"),
    "SERVICIOS": ("sector", "Servicios"),
}

CORTES_EAL20 = {
    "TOTAL": ("total", "Total"),
    "DE 5 A 9 TRABAJADORES": ("tamano_empresa", "5 a 9 trabajadores"),
    "DE 10 A 49 TRABAJADORES": ("tamano_empresa", "10 a 49 trabajadores"),
    "DE 50 A 249 TRABAJADORES": ("tamano_empresa", "50 a 249 trabajadores"),
    "DE 250 A 499 TRABAJADORES": ("tamano_empresa", "250 a 499 trabajadores"),
    "MÁS DE 499 TRABAJADORES": ("tamano_empresa", "Más de 499 trabajadores"),
}


def limpiar_texto(valor):
    # Quita espacios repetidos o finales sin perder acentos legibles.
    return re.sub(r"\s+", " ", str(valor)).strip()


def normalizar_competencia(valor):
    # Devuelve un ID estable y una etiqueta común para ambas hojas.
    original_limpio = limpiar_texto(valor)
    if original_limpio not in COMPETENCIAS:
        raise ValueError(f"Competencia sin regla de normalización: {original_limpio}")
    return COMPETENCIAS[original_limpio]


def extraer_anio(nombre_archivo):
    coincidencia = re.search(r"(20\d{2})", nombre_archivo)
    if not coincidencia:
        raise ValueError(f"No se puede extraer el año de {nombre_archivo}")
    return int(coincidencia.group(1))

## Lectura de EAL-16: importancia futura

Cada hoja contiene siete bloques. Detectamos sus títulos en vez de depender solo de números de fila; después pasamos `NADA`, `POCO`, `BASTANTE` y `MUCHO` de columnas a filas.

La columna `TOTAL` se usa como control (debe valer 100), pero no se guarda como indicador porque no aporta valor al análisis.

In [10]:
def procesar_eal16(archivo):
    anio = extraer_anio(archivo.name)
    bruto = pd.read_excel(archivo, sheet_name="EAL-16", header=None)

    # Encontramos el comienzo de cada bloque EAL-16, EAL-16a, ..., EAL-16f.
    bloques = []
    for indice, valor in bruto[0].items():
        texto = limpiar_texto(valor) if pd.notna(valor) else ""
        coincidencia = re.match(r"^(EAL-16[a-f]?)\.", texto)
        if coincidencia:
            bloques.append((indice, coincidencia.group(1)))

    if set(codigo for _, codigo in bloques) != set(CORTES_EAL16):
        raise ValueError(f"Bloques inesperados en {archivo.name}: {bloques}")

    resultados = []
    for posicion, (inicio, codigo) in enumerate(bloques):
        fin = bloques[posicion + 1][0] if posicion + 1 < len(bloques) else len(bruto)
        # Los libros completos incluyen filas en blanco variables. Buscamos la
        # cabecera por sus etiquetas dentro del bloque, no por una posición fija.
        fila_cabecera = None
        for indice in range(inicio + 1, fin):
            candidata = [
                limpiar_texto(x).upper() for x in bruto.loc[indice, 1:5] if pd.notna(x)
            ]
            if candidata == ["TOTAL", "NADA", "POCO", "BASTANTE", "MUCHO"]:
                fila_cabecera = indice
                break
        if fila_cabecera is None:
            raise ValueError(f"No se encontró la cabecera en {archivo.name}, bloque {codigo}")

        cabeceras = [limpiar_texto(x).upper() for x in bruto.loc[fila_cabecera, 1:5]]
        if cabeceras != ["TOTAL", "NADA", "POCO", "BASTANTE", "MUCHO"]:
            raise ValueError(f"Cabecera inesperada en {archivo.name}, bloque {codigo}: {cabeceras}")

        tabla = bruto.iloc[fila_cabecera + 1:fin, 0:6].copy()
        tabla.columns = ["competencia_original", *cabeceras]
        tabla = tabla[tabla["competencia_original"].notna()]

        # TOTAL debe ser 100 para todas las competencias del bloque.
        total = pd.to_numeric(tabla["TOTAL"], errors="coerce")
        if total.isna().any() or not total.between(99.99, 100.01).all():
            raise ValueError(f"TOTAL distinto de 100 en {archivo.name}, bloque {codigo}")

        desglose, corte = CORTES_EAL16[codigo]
        larga = tabla.melt(
            id_vars="competencia_original",
            value_vars=["NADA", "POCO", "BASTANTE", "MUCHO"],
            var_name="indicador",
            value_name="porcentaje",
        )
        larga["indicador"] = larga["indicador"].str.lower()
        larga["porcentaje"] = pd.to_numeric(larga["porcentaje"], errors="raise")
        larga[["competencia_id", "competencia"]] = larga["competencia_original"].apply(
            lambda x: pd.Series(normalizar_competencia(x))
        )
        larga = larga.assign(
            anio=anio,
            tema="importancia_futura",
            desglose=desglose,
            corte=corte,
            unidad="porcentaje",
            universo="Total de empresas de la categoría",
            archivo_origen=archivo.name,
            hoja_origen="EAL-16",
        )
        resultados.append(larga)

    return pd.concat(resultados, ignore_index=True)

## Lectura de EAL-20: formación por tamaño de empresa

Esta hoja presenta las competencias de formación para el total y cinco intervalos de tamaño. La pasamos a formato largo y marcamos `desglose = tamano_empresa` para distinguir su total del total calculado en la tabla sectorial.

In [11]:
def procesar_eal20(archivo):
    anio = extraer_anio(archivo.name)
    bruto = pd.read_excel(archivo, sheet_name="EAL-20", header=None)

    fila_cabecera = None
    cabeceras_esperadas = list(CORTES_EAL20)
    for indice in bruto.index:
        valores = [limpiar_texto(x).upper() for x in bruto.loc[indice, 1:6] if pd.notna(x)]
        if valores == cabeceras_esperadas:
            fila_cabecera = indice
            break
    if fila_cabecera is None:
        raise ValueError(f"No se encontró la cabecera EAL-20 en {archivo.name}")

    tabla = bruto.iloc[fila_cabecera + 1:, 0:7].copy()
    tabla.columns = ["competencia_original", *cabeceras_esperadas]
    tabla = tabla[tabla["competencia_original"].notna()]
    tabla = tabla[~tabla["competencia_original"].astype(str).str.startswith("(")]

    larga = tabla.melt(
        id_vars="competencia_original",
        value_vars=cabeceras_esperadas,
        var_name="corte_original",
        value_name="porcentaje",
    )
    larga["porcentaje"] = pd.to_numeric(larga["porcentaje"], errors="raise")
    larga["corte"] = larga["corte_original"].map(lambda x: CORTES_EAL20[x][1])
    larga[["competencia_id", "competencia"]] = larga["competencia_original"].apply(
        lambda x: pd.Series(normalizar_competencia(x))
    )
    larga = larga.assign(
        anio=anio,
        tema="formacion_ofrecida",
        desglose="tamano_empresa",
        indicador="formacion_ofrecida",
        unidad="porcentaje",
        universo="Empresas formadoras de la categoría",
        archivo_origen=archivo.name,
        hoja_origen="EAL-20",
    )
    return larga.drop(columns="corte_original")

## Lectura de EAL-21: formación ofrecida

En esta hoja las columnas representan el total y los sectores. Las convertimos al mismo esquema largo. El indicador común será `formacion_ofrecida` y el universo se identifica como empresas formadoras.

In [12]:
def procesar_eal21(archivo):
    anio = extraer_anio(archivo.name)
    bruto = pd.read_excel(archivo, sheet_name="EAL-21", header=None)

    # Buscamos la fila que contiene TOTAL, INDUSTRIA, CONSTRUCCIÓN y SERVICIOS.
    fila_cabecera = None
    for indice in bruto.index:
        valores = [limpiar_texto(x).upper() for x in bruto.loc[indice, 1:4] if pd.notna(x)]
        if valores == ["TOTAL", "INDUSTRIA", "CONSTRUCCIÓN", "SERVICIOS"]:
            fila_cabecera = indice
            break
    if fila_cabecera is None:
        raise ValueError(f"No se encontró la cabecera EAL-21 en {archivo.name}")

    cabeceras = [limpiar_texto(x).upper() for x in bruto.loc[fila_cabecera, 1:4]]
    tabla = bruto.iloc[fila_cabecera + 1:, 0:5].copy()
    tabla.columns = ["competencia_original", *cabeceras]
    tabla = tabla[tabla["competencia_original"].notna()]
    tabla = tabla[~tabla["competencia_original"].astype(str).str.startswith("(")]

    larga = tabla.melt(
        id_vars="competencia_original",
        value_vars=cabeceras,
        var_name="corte_original",
        value_name="porcentaje",
    )
    larga["porcentaje"] = pd.to_numeric(larga["porcentaje"], errors="raise")
    larga[["desglose", "corte"]] = larga["corte_original"].apply(
        lambda x: pd.Series(CORTES_EAL21[x])
    )
    larga[["competencia_id", "competencia"]] = larga["competencia_original"].apply(
        lambda x: pd.Series(normalizar_competencia(x))
    )
    larga = larga.assign(
        anio=anio,
        tema="formacion_ofrecida",
        indicador="formacion_ofrecida",
        unidad="porcentaje",
        universo="Empresas formadoras de la categoría",
        archivo_origen=archivo.name,
        hoja_origen="EAL-21",
    )
    return larga.drop(columns="corte_original")

## Unificación

Procesamos cada fichero con las dos funciones y concatenamos los resultados. Ordenamos las columnas y filas para obtener una salida determinista y fácil de comparar entre ejecuciones.

In [13]:
partes = []
for archivo in archivos_entrada:
    partes.append(procesar_eal16(archivo))
    partes.append(procesar_eal20(archivo))
    partes.append(procesar_eal21(archivo))

columnas_salida = [
    "anio", "tema", "desglose", "corte", "competencia_id", "competencia",
    "indicador", "porcentaje", "unidad", "universo", "archivo_origen", "hoja_origen",
]

datos = pd.concat(partes, ignore_index=True)[columnas_salida]
datos = datos.sort_values(
    ["anio", "tema", "desglose", "corte", "competencia_id", "indicador"]
).reset_index(drop=True)

print(f"Filas unificadas: {len(datos):,}")
print(f"Años: {sorted(datos['anio'].unique())}")
print(datos.groupby(["anio", "hoja_origen"]).size().rename("filas"))


Filas unificadas: 1,170
Años: [np.int64(2022), np.int64(2023), np.int64(2024)]
anio  hoja_origen
2022  EAL-16         280
      EAL-20          66
      EAL-21          44
2023  EAL-16         280
      EAL-20          66
      EAL-21          44
2024  EAL-16         280
      EAL-20          66
      EAL-21          44
Name: filas, dtype: int64


## Controles de calidad

Antes de exportar comprobamos los problemas que invalidarían un análisis: claves duplicadas, valores ausentes, porcentajes fuera de rango y sumas de importancia distintas de 100 por competencia.

In [14]:
clave = ["anio", "tema", "desglose", "corte", "competencia_id", "indicador"]
duplicados = datos.duplicated(clave).sum()
nulos = datos[columnas_salida].isna().sum().sum()
fuera_rango = (~datos["porcentaje"].between(0, 100)).sum()

importancia = datos[datos["tema"] == "importancia_futura"]
sumas_importancia = importancia.groupby(
    ["anio", "desglose", "corte", "competencia_id"]
)["porcentaje"].sum()
sumas_incorrectas = (~sumas_importancia.between(99.98, 100.02)).sum()

assert duplicados == 0, f"Hay {duplicados} claves duplicadas."
assert nulos == 0, f"Hay {nulos} valores ausentes."
assert fuera_rango == 0, f"Hay {fuera_rango} porcentajes fuera de [0, 100]."
assert sumas_incorrectas == 0, f"Hay {sumas_incorrectas} distribuciones que no suman 100."

print("Controles superados:")
print(f" - claves duplicadas: {duplicados}")
print(f" - valores ausentes: {nulos}")
print(f" - porcentajes fuera de rango: {fuera_rango}")
print(f" - distribuciones de importancia que no suman 100: {sumas_incorrectas}")

Controles superados:
 - claves duplicadas: 0
 - valores ausentes: 0
 - porcentajes fuera de rango: 0
 - distribuciones de importancia que no suman 100: 0


## Exportación a CSV

Guardamos con comas como separador, punto decimal y codificación `utf-8-sig`. Esta codificación conserva correctamente los acentos y facilita la apertura directa en Excel.

In [15]:
carpeta_salida.mkdir(parents=True, exist_ok=True)
datos.to_csv(
    fichero_salida,
    index=False,
    sep=",",
    decimal=".",
    encoding="utf-8-sig",
)

print(f"CSV guardado en: {fichero_salida}")
print(f"Tamaño: {fichero_salida.stat().st_size:,} bytes")

CSV guardado en: C:\Users\raulm\Desktop\Simulator\ProjecteData\Equip_33\Data\Encuesta_EAL\processed\Tablas_EAL_Competencies_2022-2024.csv
Tamaño: 229,975 bytes
